# Process MIST 2026 zen data for longer periods

In [2]:
from mth5.clients import MakeMTH5
from pathlib import Path

In [32]:
mist_folder = Path(r"c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026")
mth5_path = mist_folder / "mth5"
survey_id = "MIST2026"

## Make MTH5 with a combined run at 1s

In [34]:
station = "7404"
zm = MakeMTH5.from_zen(
    mist_folder / station,
    sample_rates=[512],
    save_path=mth5_path.joinpath(f"mist{station}.h5"),
    calibration_path=mist_folder / "antenna.cal",
    survey_id=survey_id,
    station_stem="mist",
    combine=True,
)

2026-03-16T12:49:44.828295-0700 | INFO | mth5.mth5 | _initialize_file | line: 900 | Initialized MTH5 0.2.0 file c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026\mth5\mist7404.h5 in mode w
2026-03-16T12:49:48.334772-0700 | WARNING | mth5.io.zen.zen | validate_time_blocks | line: 1455 | Skipped the first 0 seconds
2026-03-16T12:49:48.335834-0700 | WARNING | mth5.io.zen.zen | validate_time_blocks | line: 1456 | Skipped first 0 points in time series
2026-03-16T12:50:19.215268-0700 | INFO | mth5.timeseries.run_ts | _align_channels | line: 579 | Channels do not have a common start, using earliest: 2026-01-20T18:29:57.000000000
2026-03-16T12:50:19.216326-0700 | INFO | mth5.timeseries.run_ts | _align_channels | line: 584 | Channels do not have a common end, using latest: 2026-01-21T06:30:01.984375000
2026-03-16T12:50:44.558154-0700 | WARNING | mth5.io.zen.zen | validate_time_blocks | line: 1455 | Skipped the first 0 seconds
2026-03-16T12:50:44.559132-0700 | WARNING | mth5.io.zen.zen | validate

## Process Using Aurora

In [11]:
import warnings
from mtpy.processing.aurora.process_aurora import AuroraProcessing
from mt_metadata.common import MTime
from mth5.helpers import close_open_files
from mtpy import MT, MTData
import matplotlib

matplotlib.use("Agg")
from matplotlib import pyplot as plt

plt.ioff()

warnings.filterwarnings("ignore")

In [37]:
# How to combined transfer functions for the various sample rates.
merge_dict = {
    512: {"period_min": None, "period_max": 100},
    1: {"period_min": 100, "period_max": 10000},
}

sample_rates = [1]
# Stations to process
rr_geomag = True
rr_4096 = False
station_dict = {"local": "mist7404", "remote": None}
# rr_geomag_station = "Boulder"
# geomag_mth5 = Path(r"c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026\mth5\usgs_geomag_bou_xy.h5")
rr_geomag_station = "Fresno"
geomag_mth5 = Path(r"c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026\mth5\usgs_geomag_frn_xy.h5")

# band setup file. This describes which frequency bands to process at
# each decimation level.
band_file = r"c:\Users\jpeacock\OneDrive - DOI\MTData\bandset.cfg"
band_file_4096 = r"c:\Users\jpeacock\OneDrive - DOI\MTData\bandset_4096.cfg"


In [38]:
use_coherence_weighting = True

# path to store EDI files and make directory if not already exists
edi_path = mth5_path.joinpath("EDI_Files_aurora_coherence_weighting" if use_coherence_weighting else "EDI_Files_aurora")
edi_path.mkdir(exist_ok=True)

processing_dict = dict(
                [(sr, {"config": None, "kernel_dataset": None}) for sr in sample_rates]
            )

# setup AuroraProcessing object
ap = AuroraProcessing(merge_dictionary=merge_dict)
ap.local_station_id = station_dict["local"]
ap.local_mth5_path = mth5_path.joinpath(f"{ap.local_station_id}.h5")

## this will run defaults and remote reference each sample rate to
## the specified remote reference.
# tf_processed = ap.process([4096, 256, 1])

# if you want more control the you need to create a kernel dataset
# and a configuration for each sample rate.

for sample_rate in sample_rates:
    close_open_files()
    ap.df = None  # reset the dataframe
    if station_dict["remote"] is None and not rr_geomag:
        ap.remote_station_id = None
    elif sample_rate == 1 and rr_geomag:
        ap.remote_station_id = rr_geomag_station
        ap.remote_mth5_path = geomag_mth5
    else:
        ap.remote_station_id = station_dict["remote"]
        ap.remote_mth5_path = mth5_path.joinpath(
            f"{ap.remote_station_id}.h5"
        )

    # create run summary
    run_summary = ap.get_run_summary()
    run_summary.set_sample_rate(sample_rate, inplace=True)

    # create kernel dataset
    kernel_dataset = ap.create_kernel_dataset(
        run_summary=run_summary,
        local_station_id=ap.local_station_id,
        remote_station_id=ap.remote_station_id,
    )

    # drop runs that are too short
    if sample_rate == 4096:
        mimimum_run_duration = 100  # seconds
        band_setup_file = band_file_4096
    elif sample_rate == 256 or sample_rate == 512:
        mimimum_run_duration = 1000  # seconds
        band_setup_file = band_file
    elif sample_rate == 1:
        mimimum_run_duration = 3600 * 5  # seconds
        band_setup_file = band_file

    kernel_dataset.drop_runs_shorter_than(mimimum_run_duration)

    # create configuration object
    config = ap.create_config(
        kernel_dataset=kernel_dataset,
        add_coherence_weights=use_coherence_weighting,
        **{
            "emtf_band_file": band_setup_file,
            "input_channels": kernel_dataset.input_channels,
            "output_channels": kernel_dataset.output_channels,
            "save_fcs": False,
            "save_fcs_type": "h5",
        },
    )

    # add to processing dictionary
    processing_dict[sample_rate]["config"] = config
    processing_dict[sample_rate]["kernel_dataset"] = kernel_dataset

### process
processed_dict = ap.process(
    processing_dict=processing_dict,
)

# plot each TF for each sample rate
md = MTData()
for sample_rate, processed in processed_dict.items():
    if "tf" not in processed:
        print("No transfer function for {} Hz", sample_rate)
        continue
    md.add_station(processed["tf"], survey=f"sr_{sample_rate}")

p2 = md.plot_mt_response(
    list(md.keys())[::-1], plot_style="compare", fig_num=2
)
p2.save_plot(
    edi_path.joinpath(f"{ap.local_station_id}_tfs.png"),
    fig_dpi=300,
    close_plot=True,
)

# plot with MTpy
try:
    combined = processed_dict["combined"]["tf"]
except KeyError:
    raise ValueError(
        "Processing did not complete successfully. Check logs."
    )

mt_obj = MT()
mt_obj.survey_metadata = combined.survey_metadata
mt_obj._transfer_function = combined._transfer_function
mt_obj.write(edi_path.joinpath(f"{mt_obj.station}.edi"), file_type="edi")
p1 = mt_obj.plot_mt_response(fig_num=6, plot_num=2)
p1.save_plot(
    edi_path.joinpath(f"{mt_obj.station}.png"),
    fig_dpi=300,
    close_plot=True,
)

plt.close("all")
close_open_files()


2026-03-16T12:54:16.869655-0700 | INFO | mth5.mth5 | close_mth5 | line: 1035 | Flushing and closing c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026\mth5\mist7404.h5
2026-03-16T12:54:16.924199-0700 | INFO | mth5.mth5 | close_mth5 | line: 1035 | Flushing and closing c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026\mth5\usgs_geomag_frn_xy.h5
2026-03-16T12:54:18.845218-0700 | INFO | mth5.mth5 | close_mth5 | line: 1035 | Flushing and closing c:\Users\jpeacock\OneDrive - DOI\MTData\MIST2026\mth5\mist7404.h5
2026-03-16T12:54:19.016478-0700 | INFO | mtpy.processing.aurora.process_aurora | process | line: 495 | Processing sample rate 1.
2026-03-16T12:54:19.034795-0700 | ERROR | aurora.time_series.window_helpers | available_number_of_windows_in_array | line: 50 | Window is longer than the time series -- no complete windows can be returned
2026-03-16T12:54:19.035670-0700 | ERROR | aurora.time_series.window_helpers | available_number_of_windows_in_array | line: 50 | Window is longer than the time

In [23]:
ap

     survey   station       run                            start  \
0  MIST2026  mist7406  sr1_0001 2026-01-25 00:59:57.001953+00:00   

                        end       duration  
0 2026-01-26 19:29:41+00:00  152983.998047  